In [1]:
# Scientific TLDR Demo — run this cell once.
# It loads the fine-tuned BART model, downloads/loads AlignScore, and displays the interface.

import os, sys, subprocess, importlib.util, urllib.request, html, json
from pathlib import Path

# ------------------------- Project setup -------------------------
def find_project_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for candidate in candidates:
        if (candidate / "configs" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find configs/config.yaml. Open this notebook from inside your SciTLDR project "
        "or place it in the project root / notebooks folder."
    )

ROOT = find_project_root()
os.chdir(ROOT)

# Install AlignScore only if it is missing from the active notebook kernel.
def ensure_package(import_name, install_command):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {import_name} into this notebook's Python environment...")
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + install_command)

ensure_package("yaml", ["pyyaml"])
ensure_package("torch", ["torch"])
ensure_package("transformers", ["transformers"])
ensure_package("ipywidgets", ["ipywidgets"])
ensure_package("spacy", ["spacy"])
ensure_package("alignscore", ["git+https://github.com/yuh-zha/AlignScore.git"])

import yaml
import torch
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from alignscore import AlignScore

# AlignScore uses spaCy for sentence splitting.
try:
    import spacy
    spacy.load("en_core_web_sm")
except Exception:
    print("Downloading spaCy English model (one-time setup)...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

# ------------------------- Configuration -------------------------
CFG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text(encoding="utf-8"))
MAX_IN = int(CFG["data"]["max_source_len"])
GREEDY_CONFIG = dict(CFG["baseline"]["decoding"])

# Use the locally fine-tuned checkpoint if available; otherwise use the configured HF model ID.
MODEL_DIR = ROOT / "outputs" / "model"
if (MODEL_DIR / "config.json").exists():
    MODEL_SOURCE = str(MODEL_DIR)
    model_source_label = "local outputs/model"
else:
    MODEL_SOURCE = CFG["model_hub"]
    model_source_label = f"Hugging Face: {MODEL_SOURCE}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BART from {model_source_label} on {DEVICE}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)
bart = AutoModelForSeq2SeqLM.from_pretrained(MODEL_SOURCE).to(DEVICE).eval()

# ------------------------- AlignScore checkpoint -------------------------
ALIGN_DIR = ROOT / "models" / "alignscore"
ALIGN_DIR.mkdir(parents=True, exist_ok=True)
ALIGN_CKPT = ALIGN_DIR / "AlignScore-base.ckpt"
ALIGN_URL = "https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt"

if not ALIGN_CKPT.exists() or ALIGN_CKPT.stat().st_size < 1_000_000:
    print("Downloading official AlignScore-base checkpoint (one-time download)...")
    tmp_path = ALIGN_CKPT.with_suffix(".ckpt.part")
    urllib.request.urlretrieve(ALIGN_URL, tmp_path)
    tmp_path.replace(ALIGN_CKPT)

print("Loading AlignScore...")
align_scorer = AlignScore(
    model="roberta-base",
    batch_size=8,
    device=DEVICE,
    ckpt_path=str(ALIGN_CKPT),
    evaluation_mode="nli_sp"
)

# ------------------------- Generation functions -------------------------
DBS_CONFIG = {
    "num_beams": 10,
    "num_beam_groups": 5,
    "num_return_sequences": 5,
    "diversity_penalty": 1.0,
    "max_length": 64,
    "early_stopping": True,
}

def prepare_abstract(abstract):
    abstract = " ".join(abstract.split())
    if not abstract:
        raise ValueError("Please paste a scientific abstract first.")
    token_count = len(tokenizer(abstract, add_special_tokens=True)["input_ids"])
    inputs = tokenizer(
        abstract, max_length=MAX_IN, truncation=True, return_tensors="pt"
    ).to(DEVICE)
    return abstract, token_count, inputs

def run_models(abstract):
    abstract, token_count, inputs = prepare_abstract(abstract)

    with torch.inference_mode():
        # Baseline: same fine-tuned BART checkpoint, greedy decoding.
        greedy_ids = bart.generate(
            **inputs,
            **GREEDY_CONFIG
        )
        greedy_summary = tokenizer.decode(
            greedy_ids[0], skip_special_tokens=True
        ).strip()

        # Our approach: Diverse Beam Search generates five candidates.
        candidate_ids = bart.generate(**inputs, **DBS_CONFIG)

    candidates = [
        tokenizer.decode(ids, skip_special_tokens=True).strip()
        for ids in candidate_ids
    ]

    # AlignScore compares each candidate (claim) with the source abstract (context).
    scores = align_scorer.score(
        contexts=[abstract] * len(candidates),
        claims=candidates
    )
    scores = [float(s) for s in scores]
    best_idx = max(range(len(scores)), key=lambda i: scores[i])

    return {
        "abstract": abstract,
        "token_count": token_count,
        "greedy": greedy_summary,
        "candidates": candidates,
        "scores": scores,
        "selected": candidates[best_idx],
        "selected_score": scores[best_idx],
        "selected_idx": best_idx,
    }

# ------------------------- Interactive interface -------------------------
title = widgets.HTML(
    "<h2 style='margin-bottom:4px'>Scientific TLDR Generator</h2>"
    "<p style='color:#777;margin-top:0'>Generate a concise, one-sentence summary "
    "of a scientific abstract.</p>"
)
abstract_input = widgets.Textarea(
    placeholder="Paste a scientific abstract here...",
    layout=widgets.Layout(width="100%", height="190px")
)
generate_button = widgets.Button(
    description="Generate summaries",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="220px")
)
clear_button = widgets.Button(
    description="Clear",
    layout=widgets.Layout(width="100px")
)
status = widgets.HTML()
results = widgets.Output()

def on_generate(_):
    generate_button.disabled = True
    status.value = "<p style='color:#777'>Generating summaries and scoring candidates…</p>"
    with results:
        clear_output(wait=True)
        try:
            result = run_models(abstract_input.value)
            if result["token_count"] > MAX_IN:
                print(
                    f"Note: input contains {result['token_count']} tokens; "
                    f"BART reads the first {MAX_IN} tokens."
                )
            display(HTML(
                "<h3>Greedy BART — Baseline</h3>"
                f"<div style='padding:12px;border:1px solid #ccc;border-radius:8px'>"
                f"{html.escape(result['greedy'])}</div>"
            ))
            display(HTML(
                "<h3>DBS + AlignScore — Selected Summary</h3>"
                f"<div style='padding:12px;border:2px solid #39865b;border-radius:8px'>"
                f"<b>{html.escape(result['selected'])}</b><br>"
                f"<small>AlignScore: {result['selected_score']:.4f} "
                f"· Candidate {result['selected_idx'] + 1} of 5</small></div>"
            ))
            display(HTML("<h3>All DBS Candidates</h3>"))
            rows = []
            for i, (candidate, score) in enumerate(zip(result["candidates"], result["scores"])):
                marker = " ✓ Selected" if i == result["selected_idx"] else ""
                rows.append(
                    f"<tr><td style='padding:8px;border-bottom:1px solid #ddd'>{i+1}{marker}</td>"
                    f"<td style='padding:8px;border-bottom:1px solid #ddd'>{html.escape(candidate)}</td>"
                    f"<td style='padding:8px;border-bottom:1px solid #ddd;text-align:right'>{score:.4f}</td></tr>"
                )
            display(HTML(
                "<table style='width:100%;border-collapse:collapse'>"
                "<thead><tr><th align='left'>#</th><th align='left'>Candidate</th>"
                "<th align='right'>AlignScore</th></tr></thead><tbody>"
                + "".join(rows) + "</tbody></table>"
            ))
            status.value = "<p style='color:green'>Done.</p>"
        except Exception as exc:
            status.value = "<p style='color:#b33'>Generation failed. See error below.</p>"
            raise
        finally:
            generate_button.disabled = False

def on_clear(_):
    abstract_input.value = ""
    status.value = ""
    with results:
        clear_output()

generate_button.on_click(on_generate)
clear_button.on_click(on_clear)

display(title, abstract_input, widgets.HBox([generate_button, clear_button]), status, results)

# ------------------------- Reported held-out test results -------------------------
display(HTML("""
<hr>
<h3>Final Test Results (615 evaluated examples)</h3>
<table style="border-collapse:collapse;width:100%">
<tr><th align="left" style="padding:7px">Metric</th><th align="right" style="padding:7px">Greedy BART</th>
<th align="right" style="padding:7px">DBS + AlignScore (penalty 1.0)</th><th align="right" style="padding:7px">Difference</th></tr>
<tr><td style="padding:7px">SummaC</td><td align="right">0.7612</td><td align="right">0.8753</td><td align="right">+0.1141</td></tr>
<tr><td style="padding:7px">ROUGE-L</td><td align="right">0.3675</td><td align="right">0.3658</td><td align="right">-0.0017</td></tr>
</table>
<p style="color:#777;font-size:0.9em">These are the previously reported aggregate test results, not scores computed for the text entered above.</p>
"""))

print("Demo ready. BART and AlignScore are loaded; use the text box above.")

W1112 08:52:06.665000 5472 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading BART from Hugging Face: SamHuertas/SciTLDR-trained-BART on cuda...
Loading AlignScore...


Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\admin\Documents\Repositories\csc173-midterm\models\alignscore\AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\admin\Documents\Repositories\csc173-midterm\.venv-scoring\Lib\site-packages\pytorch_lightning\core\saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['base_model.embeddings.position_ids']


HTML(value="<h2 style='margin-bottom:4px'>Scientific TLDR Generator</h2><p style='color:#777;margin-top:0'>Gen…

Textarea(value='', layout=Layout(height='190px', width='100%'), placeholder='Paste a scientific abstract here.…

HTML(value='')

Output()

Metric,Greedy BART,DBS + AlignScore (penalty 1.0),Difference
SummaC,0.7612,0.8753,+0.1141
ROUGE-L,0.3675,0.3658,-0.0017


Demo ready. BART and AlignScore are loaded; use the text box above.
